#### Downsampling Point cloud

In [ ]:
import os
import pandas as pd
import torch
import sys
import glob
sys.path.append('./')  # make sure utils.misc is in your project path
from utils.misc import fps

# Input and output directories
source_dir = 'dataset/pointclouds_8192'   # original point clouds (8192 points)
target_dir = 'dataset/pointclouds_2048'   # downsampled point clouds (2048 points)
sample_size = 2048

os.makedirs(target_dir, exist_ok=True)

# Find all CSV files recursively
csv_files = sorted(glob.glob(os.path.join(source_dir, '**', '*.csv'), recursive=True))

for file in csv_files:
    print(f'Processing: {file}')
    df = pd.read_csv(file)
    lw_df = df[["X", "Y", "Z"]]

    # Convert to torch tensor
    lw_points = torch.tensor(lw_df.values).cuda().float().contiguous().unsqueeze(0)

    # Apply FPS downsampling if sufficient points exist
    if lw_points.shape[1] >= sample_size:
        sampled_points = fps(lw_points, sample_size)
        sampled_points = sampled_points.cpu().numpy().squeeze()
    else:
        sampled_points = lw_points.cpu().numpy().squeeze()  # fallback for very small files
    
    sampled_df = pd.DataFrame(sampled_points, columns=["X", "Y", "Z"])

    # Keep original folder structure in the target directory
    rel_path = os.path.relpath(file, source_dir)  
    save_path = os.path.join(target_dir, rel_path)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    sampled_df.to_csv(save_path, index=False)

    print(f"Saved: {save_path} ({sampled_df.shape[0]} points)")


#### ModelNet hdf generation

In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

###########################
# 0. Define label map
###########################
# ModelNet40 classes + "tree" (custom addition)
modelnet40_classes = [
    "airplane", "bathtub", "bed", "bench", "bookshelf", "bottle", "bowl",
    "car", "chair", "cone", "cup", "curtain", "desk", "door", "dresser",
    "flower_pot", "glass_box", "guitar", "keyboard", "lamp", "laptop",
    "mantel", "monitor", "night_stand", "person", "piano", "plant", "radio",
    "range_hood", "sink", "sofa", "stairs", "stool", "table", "tent",
    "toilet", "tv_stand", "vase", "wardrobe", "xbox", "tree"
]
class2label = {c: i for i, c in enumerate(modelnet40_classes)}  # e.g., airplane=0, tree=40


###############################################
# 1. Load ModelNet40 HDF5 (data, label arrays)
###############################################
def load_modelnet_data(h5_dir, partition='train'):
    all_data, all_label = [], []
    h5_files = sorted(glob.glob(os.path.join(h5_dir, f'ply_data_{partition}*.h5')))
    for h5_name in h5_files:
        with h5py.File(h5_name, 'r') as f:
            data = f['data'][:]
            label = f['label'][:]
        all_data.append(data)
        all_label.append(label)

    if len(all_data) == 0:
        return None, None

    all_data = np.concatenate(all_data, axis=0)
    all_label = np.concatenate(all_label, axis=0)

    if len(all_label.shape) == 2:
        all_label = all_label.squeeze()
    return all_data, all_label


###############################################
# 2. Load CSV point cloud data (2048×3 + label)
###############################################
def load_csv_data(csv_dir, category):
    csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))
    csv_data_list, csv_label_list = [], []

    label_id = class2label[category]

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, header=None)

        # Remove first row if it contains string headers like "X, Y, Z"
        df = df[1:]
        arr = df.values.astype(np.float32)

        if arr.shape != (2048, 3):
            print(f"Warning: {csv_file} shape {arr.shape}, expected (2048,3). Skipped.")
            continue

        csv_data_list.append(arr)
        csv_label_list.append(label_id)

    if len(csv_data_list) == 0:
        return None, None

    csv_data = np.stack(csv_data_list, axis=0)
    csv_label = np.array(csv_label_list, dtype=np.int64)
    return csv_data, csv_label


###############################################
# 3. Merge ModelNet and custom CSV datasets
###############################################
def merge_datasets(modelnet_data, modelnet_label, csv_data_dict):
    csv_train_data_list, csv_test_data_list = [], []
    csv_train_label_list, csv_test_label_list = [], []

    for category, (csv_data, csv_label) in csv_data_dict.items():
        if csv_data is not None and csv_data.shape[0] > 0:
            csv_train_data, csv_test_data, csv_train_label, csv_test_label = train_test_split(
                csv_data, csv_label, test_size=0.2, random_state=42
            )
            csv_train_data_list.append(csv_train_data)
            csv_test_data_list.append(csv_test_data)
            csv_train_label_list.append(csv_train_label)
            csv_test_label_list.append(csv_test_label)

    train_data = np.concatenate([modelnet_data.get('train', np.empty((0, 2048, 3), dtype=np.float32))] + csv_train_data_list, axis=0)
    train_label = np.concatenate([modelnet_label.get('train', np.empty((0,), dtype=np.int64))] + csv_train_label_list, axis=0)
    test_data = np.concatenate([modelnet_data.get('test', np.empty((0, 2048, 3), dtype=np.float32))] + csv_test_data_list, axis=0)
    test_label = np.concatenate([modelnet_label.get('test', np.empty((0,), dtype=np.int64))] + csv_test_label_list, axis=0)

    return train_data, train_label, test_data, test_label


###############################################
# 4. Save merged data to HDF5 (chunked)
###############################################
def save_h5_chunked(data, label, out_prefix, chunk_size=2048):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    N = data.shape[0]
    num_chunks = (N + chunk_size - 1) // chunk_size

    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, N)
        chunk_data = data[start:end]
        chunk_label = label[start:end]
        filename = f"{out_prefix}_{i}.h5"
        with h5py.File(filename, 'w') as f:
            f.create_dataset('data', data=chunk_data, compression='gzip', compression_opts=4)
            f.create_dataset('label', data=chunk_label, compression='gzip', compression_opts=4)
        print(f"Saved {filename} with {chunk_data.shape[0]} samples")


###############################################
# 5. Main execution example
###############################################
if __name__ == "__main__":
    modelnet_h5_dir = "dataset/modelnet40_hdf5_2048"
    csv_dirs = {
        "tree": "dataset/custom_tree_csv",   # your custom category folder
    }
    out_dir = "outputs/modelnet_tree_h5"
    os.makedirs(out_dir, exist_ok=True)

    # Load ModelNet40
    train_data_m, train_label_m = load_modelnet_data(modelnet_h5_dir, partition='train')
    test_data_m, test_label_m = load_modelnet_data(modelnet_h5_dir, partition='test')

    # Load CSV datasets
    csv_data_dict = {cat: load_csv_data(dir, cat) for cat, dir in csv_dirs.items()}

    # Merge datasets
    merged_train_data, merged_train_label, merged_test_data, merged_test_label = merge_datasets(
        {'train': train_data_m, 'test': test_data_m},
        {'train': train_label_m, 'test': test_label_m},
        csv_data_dict
    )

    # Save to HDF5
    save_h5_chunked(merged_train_data, merged_train_label, os.path.join(out_dir, "ply_data_train_tree"), chunk_size=2048)
    save_h5_chunked(merged_test_data, merged_test_label, os.path.join(out_dir, "ply_data_test_tree"), chunk_size=2048)
